# 🚀 Portkey AI — Comprehensive Testing Notebook

This notebook consolidates all testing scenarios for the **Portkey AI** platform, including:

| Section | Description |
|---|---|
| [1. Setup](#1-setup) | Install dependencies and configure environment |
| [2. Basic Chat Completion](#2-basic-chat-completion) | Simple SDK usage with OpenRouter virtual key |
| [3. Chat Completion with Guardrails](#3-chat-completion-with-inline-guardrail-config) | Inline config with regex guardrail hooks |
| [4. Guardrail Tests](#4-guardrail-tests) | `default.regexMatch` — `rule` vs `regex` key behavior |
| [5. Self-Hosted Gateway](#5-self-hosted-gateway-tests) | Direct REST calls to a local Portkey Gateway instance |
| [6. MCP Server](#6-mcp-server-fastmcp) | FastMCP server — tools, resources, prompts |
| [7. MCP Registry](#7-mcp-registry) | Register a custom MCP server with Portkey |
| [8. MCP Listing](#8-mcp-listing) | List all registered MCP integrations |
| [9. Fallback Strategy](#9-fallback-strategy) | Auto-switch to backup LLMs on failure |
| [10. Automatic Retries](#10-automatic-retries) | Exponential backoff retries with custom status codes |
| [11. Caching](#11-caching-simple--semantic) | Simple & semantic cache, force refresh, namespaces |
| [12. Load Balancing](#12-load-balancing) | Weighted traffic distribution across providers |
| [13. Conditional Routing](#13-conditional-routing) | Route by metadata, params, or URL path |
| [14. Canary Testing](#14-canary-testing) | Test new models on small % of traffic |
| [15. Combined Strategies](#15-combined--nested-strategies) | Nested fallback + LB + retry + cache configs |
| [16. Guardrails + Configs](#16-guardrails-with-configs-advanced-patterns) | Retry/fallback on guardrail fail (246/446) |
| [17. Metadata & Tracing](#17-metadata--tracing) | Custom tags, trace_id correlation, user tracking |
| [18. Streaming](#18-streaming-chat-completions) | SSE streaming with fallback support |
| [19. Virtual Keys](#19-virtual-keys--model-catalog) | List and manage provider virtual keys |
| [20. Feedback API](#20-feedback-api) | Attach feedback to requests via trace_id |
| [21. Prompt Templates](#21-prompt-templates-prompt-api) | Prompt Engineering Studio — list, render, complete |

> **Prerequisites:** Python 3.9+, a Portkey account, and API keys configured in `.env`

---
## 1. Setup

Install all required packages and load environment variables.

### Required packages
```
portkey-ai      — Portkey Python SDK
fastmcp         — FastMCP server framework
python-dotenv   — Load .env files
httpx           — Async HTTP client (used by MCP server)
requests        — Sync HTTP client (used by gateway tests)
pydantic        — Data validation (used by MCP server)
```

### `.env` file format
Create a `.env` file in this directory with:
```
PORTKEY_API_KEY=your_portkey_api_key_here
```
All other API keys (Exchange, Aviation, Weather) are embedded in the MCP server section for demonstration purposes.

In [1]:
# Install required dependencies
%pip install portkey-ai fastmcp python-dotenv httpx requests pydantic pytest -q


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file in this directory
load_dotenv()

PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY")

if not PORTKEY_API_KEY:
    print("⚠️  WARNING: PORTKEY_API_KEY not found in environment.")
    print("   Create a .env file with: PORTKEY_API_KEY=your_key_here")
else:
    masked = PORTKEY_API_KEY[:6] + "*" * (len(PORTKEY_API_KEY) - 6)
    print(f"✅ PORTKEY_API_KEY loaded: {masked}")

---
## 2. Basic Chat Completion

The simplest usage of the Portkey SDK — create a client with your API key and a virtual key pointing to a provider, then call `chat.completions.create`.

### How it works
- `@hk-openrouter` is a **Portkey Virtual Key** that wraps your OpenRouter credentials
- `model` is prefixed with the virtual key slug to route to the correct provider
- The response is OpenAI-compatible

> 📌 Virtual keys let you swap providers without changing application code.

In [2]:
from portkey_ai import Portkey

# Initialize the Portkey client
portkey = Portkey(api_key=PORTKEY_API_KEY)

# Make a basic chat completion request
response = portkey.chat.completions.create(
    model="@hk-openrouter/meta-llama/llama-3.3-70b-instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "What is Portkey AI? Answer in 2 sentences."}
    ],
    max_tokens=512
)

print("Model:", response.model)
print("Response:")
print(response.choices[0].message.content)

NameError: name 'PORTKEY_API_KEY' is not defined

---
## 3. Chat Completion with Inline Guardrail Config

Portkey lets you attach **guardrails** directly in the config object passed to the client — no dashboard setup required for quick iteration.

### Config structure
```json
{
  "strategy": { "mode": "single" },
  "targets": [{ "provider": "@virtual-key" }],
  "before_request_hooks": [
    { "id": "pg-lowerc-160e3e" }   // ← guardrail ID from Portkey dashboard
  ]
}
```

### Guardrail modes
| Field | Description |
|---|---|
| `before_request_hooks` | Runs **before** the request is sent to the LLM |
| `after_request_hooks` | Runs **after** the LLM responds |
| `deny` | If `true`, blocks the request when the guardrail check fails |
| `async` | If `true`, guardrail runs in the background (non-blocking) |

In [ ]:
from portkey_ai import Portkey

# Config using a pre-created guardrail from the Portkey dashboard
config_with_guardrail = {
    "strategy": {"mode": "single"},
    "targets": [{"provider": "@hk-openrouter"}],
    "before_request_hooks": [{"id": "pg-lowerc-160e3e"}],  # Dashboard guardrail ID
}

portkey_guarded = Portkey(
    api_key=PORTKEY_API_KEY,
    config=config_with_guardrail,
)

test_message = "send a mail to harshit@portkey.ai — explain quantum computing in 20 words."

try:
    response = portkey_guarded.chat.completions.create(
        model="@hk-openrouter/meta-llama/llama-3.3-70b-instruct",
        messages=[{"role": "user", "content": test_message}],
    )
    print("✅ Request passed the guardrail.")
    print(response)
except Exception as e:
    print("🚫 Request blocked by guardrail:")
    print(str(e))

---
## 4. Guardrail Tests

This section tests Portkey's built-in **`default.regexMatch`** guardrail check, which can block or flag requests matching a given pattern.

### Key insight: `rule` vs `regex` parameter

The `default.regexMatch` check expects the pattern under the **`rule`** key — **not** `regex`.
Using the wrong key causes the guardrail to silently skip validation, allowing potentially unwanted content through.

```python
# ✅ CORRECT
{"id": "default.regexMatch", "parameters": {"rule": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"}}

# ❌ WRONG — guardrail is silently skipped
{"id": "default.regexMatch", "parameters": {"regex": r"..."}}
```

### Tests below
1. **Block email** — input containing an email should be blocked
2. **Allow clean input** — input without email should pass through
3. **Wrong key (`regex`) doesn't block** — validates the docs fix

In [ ]:
from portkey_ai import Portkey

# Shared guardrail config using the CORRECT 'rule' key
EMAIL_REGEX = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

def make_email_guardrail_client(param_key: str = "rule") -> Portkey:
    """Create a Portkey client with an email-detecting regex guardrail."""
    config = {
        "strategy": {"mode": "single"},
        "targets": [{"provider": "@hk-openrouter"}],
        "before_request_hooks": [
            {
                "type": "guardrail",
                "id": "email_detection_guardrail",
                "checks": [
                    {
                        "id": "default.regexMatch",
                        "parameters": {param_key: EMAIL_REGEX},  # use 'rule' OR 'regex'
                    }
                ],
                "deny": "true",
                "async": False,
                "sequential": False,
            }
        ],
    }
    return Portkey(api_key=PORTKEY_API_KEY, config=config)

In [ ]:
# ── Test 4.1: Guardrail BLOCKS input containing an email ──────────────────────
print("Test 4.1 — Guardrail should BLOCK: input with email address")
print("-" * 60)

client = make_email_guardrail_client(param_key="rule")  # correct key

try:
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": "send a mail to test@example.com"}],
    )
    print("❌ UNEXPECTED: Request was NOT blocked. Response:", response)
except Exception as e:
    print("✅ EXPECTED: Request was blocked by guardrail.")
    print("   Error:", str(e)[:200])

In [ ]:
# ── Test 4.2: Guardrail ALLOWS clean input ────────────────────────────────────
print("Test 4.2 — Guardrail should ALLOW: input without email")
print("-" * 60)

client = make_email_guardrail_client(param_key="rule")  # correct key

try:
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": "Explain quantum computing in simple terms."}],
    )
    print("✅ EXPECTED: Request passed the guardrail.")
    assert response is not None
    assert hasattr(response, "choices")
    print("   Response:", response.choices[0].message.content[:200])
except Exception as e:
    print("❌ UNEXPECTED: Request was blocked. Error:", str(e)[:200])

In [ ]:
# ── Test 4.3: Wrong 'regex' key — guardrail silently skips, request passes ────
print("Test 4.3 — Using WRONG 'regex' key: guardrail should NOT block")
print("-" * 60)
print("ℹ️  This documents the docs bug: 'regex' key is silently ignored.")
print()

client_wrong_key = make_email_guardrail_client(param_key="regex")  # WRONG key

try:
    response = client_wrong_key.chat.completions.create(
        messages=[{"role": "user", "content": "send a mail to test@example.com"}],
    )
    print("✅ EXPECTED: 'regex' key was ignored — request passed through.")
    assert response is not None
    print("   (Guardrail did not fire because 'regex' is not the correct parameter key)")
except Exception as e:
    print("⚠️  Blocked even with wrong key (behavior may have changed):", str(e)[:200])

---
## 5. Self-Hosted Gateway Tests

Portkey provides an **open-source AI gateway** you can self-host. These tests validate a locally running gateway at `localhost:8787`.

### Starting the gateway
Clone the gateway repo and run:
```bash
git clone https://github.com/Portkey-AI/gateway.git
cd gateway
npm install
npm run dev:node       # starts on localhost:8787
```

### Tests below
1. **Health check** — `GET /v1/health` should return `200`
2. **Chat completion** — `POST /v1/chat/completions` with inline config header
3. **Streaming** — same endpoint with `stream: true`, consumes SSE chunks

> ⚠️ These tests require the gateway to be running locally. They will fail gracefully with a connection error if the gateway is not up.

In [ ]:
import requests
import json

GATEWAY_URL = "http://localhost:8787"
GATEWAY_API_KEY = PORTKEY_API_KEY  # same key, or use a dedicated gateway key

In [ ]:
# ── Test 5.1: Health Check ────────────────────────────────────────────────────
print("Test 5.1 — Gateway Health Check")
print("-" * 60)

try:
    resp = requests.get(f"{GATEWAY_URL}/v1/health", timeout=5)
    print(f"Status: {resp.status_code}")
    print(f"Response: {json.dumps(resp.json(), indent=2)}")
    if resp.status_code == 200:
        print("✅ Gateway is healthy.")
    else:
        print("⚠️  Gateway returned non-200.")
except requests.ConnectionError:
    print("❌ Cannot connect to gateway at", GATEWAY_URL)
    print("   Start it with: npm run dev:node (inside the gateway repo)")

In [ ]:
# ── Test 5.2: Chat Completion via Gateway ─────────────────────────────────────
print("Test 5.2 — Chat Completion via Self-Hosted Gateway")
print("-" * 60)

# The guardrail/config can be passed as a JSON-encoded header
inline_config = {"before_request_hooks": ["pg-hk-low-c59982"]}

headers = {
    "Content-Type": "application/json",
    "x-portkey-api-key": GATEWAY_API_KEY,
    "x-portkey-config": json.dumps(inline_config),
}

payload = {
    "model": "@vertex-new/gemini-1.5-flash-001",
    "messages": [{"role": "user", "content": "Say hello in one sentence."}],
    "max_tokens": 100,
}

try:
    resp = requests.post(
        f"{GATEWAY_URL}/v1/chat/completions",
        headers=headers,
        json=payload,
        timeout=30,
    )
    print(f"Status: {resp.status_code}")
    data = resp.json()
    print(json.dumps(data, indent=2))
    if resp.status_code == 200 and "choices" in data:
        print("\n✅ Assistant:", data["choices"][0]["message"]["content"])
except requests.ConnectionError:
    print("❌ Cannot connect to gateway at", GATEWAY_URL)

In [ ]:
# ── Test 5.3: Streaming via Gateway ───────────────────────────────────────────
print("Test 5.3 — Streaming Chat Completion via Self-Hosted Gateway")
print("-" * 60)

headers_stream = {
    "Content-Type": "application/json",
    "x-portkey-api-key": GATEWAY_API_KEY,
}

payload_stream = {
    "model": "@hk-gemini-test-dev/gemini-1.5-flash-001",
    "messages": [{"role": "user", "content": "Count from 1 to 5."}],
    "max_tokens": 100,
    "stream": True,
}

try:
    resp = requests.post(
        f"{GATEWAY_URL}/v1/chat/completions",
        headers=headers_stream,
        json=payload_stream,
        timeout=30,
        stream=True,
    )
    print(f"Status: {resp.status_code}")

    if resp.status_code != 200:
        print("Error:", resp.text)
    else:
        print("Streamed response: ", end="", flush=True)
        for line in resp.iter_lines():
            if not line:
                continue
            decoded = line.decode("utf-8")
            if decoded.startswith("data: ") and decoded != "data: [DONE]":
                chunk = json.loads(decoded[6:])
                delta = chunk.get("choices", [{}])[0].get("delta", {})
                content = delta.get("content", "")
                print(content, end="", flush=True)
        print()
        print("\n✅ Stream complete.")
except requests.ConnectionError:
    print("❌ Cannot connect to gateway at", GATEWAY_URL)

---
## 6. MCP Server (FastMCP)

This section defines a **FastMCP server** exposing multiple real-world tools, a resource, and a prompt template.

### Architecture
```
FastMCP Server (port 8000)
├── Tools
│   ├── add_numbers          — Basic arithmetic
│   ├── get_current_time     — Server timestamp
│   ├── generate_random_number
│   ├── get_exchange_rates   — ExchangeRate-API (with TTL cache)
│   ├── convert_currency     — Currency conversion
│   ├── weather_forecast     — WeatherAPI.com
│   ├── get_airports         — AviationStack airports
│   ├── get_live_flights     — AviationStack live flights
│   └── search_in_memory     — Context-aware search
├── Resources
│   └── system_info          — Python version, OS, hostname
└── Prompts
    └── analyze_data         — Data analysis prompt template
```

### Running the server
Run this cell to **define** the server. To actually start it:
```bash
python test_mcp_server.py
# Server starts at http://0.0.0.0:8000/mcp
```

> ⚠️ You cannot start a long-running HTTP server inside a Jupyter notebook cell — run it from the terminal. The cell below defines the server for reference and import purposes.

In [ ]:
# ── MCP Server Definition ─────────────────────────────────────────────────────
# This cell defines the full MCP server.
# To run it, execute this file directly: python test_mcp_server.py
# Or call mcp.run(...) from a separate terminal.

from fastmcp import FastMCP
from fastmcp.server import Context

import random
import datetime
import logging
import platform
import time
import httpx
from typing import Dict, Any
from pydantic import BaseModel, Field

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("mcp-server")

# ── Server Init ───────────────────────────────────────────────────────────────
mcp = FastMCP("My Awesome MCP Server")

# ── API Keys / Base URLs ──────────────────────────────────────────────────────
EXCHANGE_API_KEY   = "9e233f662d770df215d4d0fa"
EXCHANGE_BASE_URL  = "https://v6.exchangerate-api.com/v6"

AVIATION_BASE_URL  = "https://api.aviationstack.com/v1"
AVIATION_ACCESS_KEY = "0606263b53a77f989275b3776200f726"

WEATHER_API_KEY    = "ea352def6e294027b4d93526260604"
WEATHER_BASE_URL   = "https://api.weatherapi.com/v1"

# ── TTL Cache ─────────────────────────────────────────────────────────────────
_CACHE: Dict = {}
TTL = 60  # seconds

def get_cache(key):
    if key in _CACHE:
        value, ts = _CACHE[key]
        if time.time() - ts < TTL:
            return value
    return None

def set_cache(key, value):
    _CACHE[key] = (value, time.time())

# ── Pydantic Models ───────────────────────────────────────────────────────────
class ExchangeRatesResponse(BaseModel):
    result: str
    time_last_update_utc: str
    time_next_update_utc: str
    base_code: str
    conversion_rates: Dict[str, float]

class ConvertRequest(BaseModel):
    from_currency: str = Field(..., min_length=3, max_length=3)
    to_currency:   str = Field(..., min_length=3, max_length=3)
    amount:        float = Field(..., gt=0)

# ── Exchange Service ──────────────────────────────────────────────────────────
class ExchangeService:
    async def get_rates(self, base: str) -> ExchangeRatesResponse:
        url = f"{EXCHANGE_BASE_URL}/{EXCHANGE_API_KEY}/latest/{base.upper()}"
        cached = get_cache(url)
        if cached:
            return cached
        async with httpx.AsyncClient(timeout=10.0) as client:
            response = await client.get(url)
            response.raise_for_status()
            data = response.json()
        if data.get("result") != "success":
            raise Exception("Exchange API failed")
        parsed = ExchangeRatesResponse(**data)
        set_cache(url, parsed)
        return parsed

exchange_service = ExchangeService()

print("✅ MCP server defined. Tools, resources, and prompts registered below.")

In [ ]:
# ── Utility Tools ─────────────────────────────────────────────────────────────

@mcp.tool
def add_numbers(a: int, b: int) -> int:
    """Add two integers."""
    logger.info(f"[add_numbers] a={a}, b={b}")
    return a + b

@mcp.tool
def get_current_time() -> str:
    """Return the current server time as a formatted string."""
    logger.info("[get_current_time]")
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@mcp.tool
def generate_random_number(min_val: int = 1, max_val: int = 100) -> int:
    """Generate a random integer between min_val and max_val (inclusive)."""
    logger.info(f"[generate_random_number] min_val={min_val}, max_val={max_val}")
    if min_val > max_val:
        raise ValueError("min_val cannot be greater than max_val")
    return random.randint(min_val, max_val)

print("✅ Utility tools registered: add_numbers, get_current_time, generate_random_number")

In [ ]:
# ── Currency Tools ────────────────────────────────────────────────────────────

@mcp.tool
async def get_exchange_rates(base_currency: str) -> dict:
    """Fetch the latest exchange rates for a given base currency (e.g. USD, EUR, INR)."""
    logger.info(f"[get_exchange_rates] base_currency={base_currency}")
    data = await exchange_service.get_rates(base_currency.upper())
    return {
        "base": data.base_code,
        "rates": data.conversion_rates,
        "last_updated": data.time_last_update_utc,
    }

@mcp.tool
async def convert_currency(input: ConvertRequest) -> dict:
    """Convert an amount from one currency to another."""
    logger.info(f"[convert_currency] input={input}")
    data = await exchange_service.get_rates(input.from_currency.upper())
    rate = data.conversion_rates.get(input.to_currency.upper())
    if not rate:
        raise ValueError("Unsupported currency")
    return {
        "converted_amount": rate * input.amount,
        "rate": rate,
    }

print("✅ Currency tools registered: get_exchange_rates, convert_currency")

In [ ]:
# ── Weather Tools ─────────────────────────────────────────────────────────────

async def fetch_weather(endpoint: str, params: Dict[str, Any]) -> Dict:
    """Internal helper: fetch data from WeatherAPI.com."""
    logger.info(f"[fetch_weather] endpoint={endpoint}, params={params}")
    params["key"] = WEATHER_API_KEY
    url = f"{WEATHER_BASE_URL}/{endpoint}"
    async with httpx.AsyncClient() as client:
        response = await client.get(url, params=params)
        response.raise_for_status()
        return response.json()

@mcp.tool
async def weather_forecast(location: str, days: int = 3) -> Dict:
    """Get a weather forecast for a city. days: 1–10."""
    logger.info(f"[weather_forecast] location={location}, days={days}")
    return await fetch_weather("forecast.json", {"q": location, "days": min(days, 10)})

print("✅ Weather tool registered: weather_forecast")

In [ ]:
# ── Aviation Tools ────────────────────────────────────────────────────────────

@mcp.tool
async def get_airports(country_code: str = None, limit: int = 10):
    """List airports, optionally filtered by ISO 2-letter country code (e.g. 'IN', 'US')."""
    logger.info(f"[get_airports] country_code={country_code}, limit={limit}")
    params = {"access_key": AVIATION_ACCESS_KEY}
    if country_code:
        params["country_code"] = country_code
    url = f"{AVIATION_BASE_URL}/airports"
    async with httpx.AsyncClient() as client:
        response = await client.get(url, params=params)
        response.raise_for_status()
        data = response.json()
    return data.get("data", [])[:limit]

@mcp.tool
async def get_live_flights(limit: int = 10):
    """Fetch currently airborne live flights (non-ground, live tracking)."""
    logger.info(f"[get_live_flights] limit={limit}")
    params = {"access_key": AVIATION_ACCESS_KEY}
    url = f"{AVIATION_BASE_URL}/flights"
    async with httpx.AsyncClient() as client:
        response = await client.get(url, params=params)
        response.raise_for_status()
        data = response.json()
    flights = []
    for flight in data.get("data", []):
        live = flight.get("live")
        if live and not live.get("is_ground"):
            flights.append({
                "airline":       flight.get("airline", {}).get("name"),
                "flight_number": flight.get("flight", {}).get("iata"),
                "departure":     flight.get("departure", {}).get("iata"),
                "arrival":       flight.get("arrival", {}).get("iata"),
            })
        if len(flights) >= limit:
            break
    return flights

print("✅ Aviation tools registered: get_airports, get_live_flights")

In [ ]:
# ── Resource & Prompt ─────────────────────────────────────────────────────────

@mcp.resource(name="system_info", uri="resource://system_info")
def get_system_info() -> dict:
    """Expose system metadata as an MCP resource."""
    logger.info("[get_system_info]")
    return {
        "python_version": platform.python_version(),
        "os": platform.system(),
        "hostname": platform.node(),
    }

@mcp.prompt
def analyze_data(data: str) -> str:
    """Prompt template for data analysis."""
    logger.info(f"[analyze_data] data={data}")
    return f"Analyze the following data:\n{data}"

@mcp.tool
def search_in_memory(query: str, ctx: Context) -> str:
    """Demo tool that uses MCP Context for logging/progress."""
    logger.info(f"[search_in_memory] query={query}")
    return f"Searching for '{query}'..."

print("✅ Resource registered: system_info")
print("✅ Prompt registered:   analyze_data")
print("✅ Context tool registered: search_in_memory")
print()
print("📋 All tools:")
print("   To start the server, run from terminal:")
print("   python test_mcp_server.py")
print("   → Server available at http://0.0.0.0:8000/mcp")

---
## 7. MCP Registry

Once your MCP server is running, **register it with Portkey** so that Portkey-connected LLMs can discover and call its tools.

### Registration payload
```json
{
  "name": "My Awesome MCP Server",
  "url": "http://0.0.0.0:8000/mcp",
  "auth_type": "none",
  "transport": "http"
}
```

### Notes
- The `url` must be publicly reachable if your Portkey account is cloud-hosted
- For local development, use `ngrok` or similar tunneling: `ngrok http 8000`
- `auth_type` can be `"none"`, `"api_key"`, or `"bearer"` depending on your server's auth

In [ ]:
import requests
import json

PORTKEY_BASE_URL = "https://api.portkey.ai/v1"

# ── Register MCP Integration ──────────────────────────────────────────────────
print("Registering MCP server with Portkey...")
print("-" * 60)

payload = {
    "name": "My Awesome MCP Server",
    "url": "http://0.0.0.0:8000/mcp",  # Change to your public URL if cloud-hosted
    "auth_type": "none",
    "transport": "http",
}

headers = {
    "x-portkey-api-key": PORTKEY_API_KEY,
    "Content-Type": "application/json",
}

try:
    response = requests.post(
        f"{PORTKEY_BASE_URL}/mcp-integrations",
        json=payload,
        headers=headers,
        timeout=30,
    )
    result = response.json()
    print(f"Status Code: {response.status_code}")
    print(json.dumps(result, indent=2))

    if response.status_code in (200, 201):
        print("\n✅ MCP server registered successfully!")
        integration_id = result.get("id") or result.get("data", {}).get("id")
        if integration_id:
            print(f"   Integration ID: {integration_id}")
    else:
        print("\n⚠️  Registration returned non-success status.")
except Exception as e:
    print("❌ Error:", str(e))

---
## 8. MCP Listing

Retrieve all MCP integrations currently registered in your Portkey workspace.

### API endpoint
```
GET https://api.portkey.ai/v1/mcp-integrations?page_size=100
```

The response is paginated. Increase `page_size` (up to 500) to retrieve more integrations at once.

In [ ]:
import requests
import json

# ── List All MCP Integrations ─────────────────────────────────────────────────
print("Fetching MCP integrations from Portkey...")
print("-" * 60)

page_size = 100
url = f"{PORTKEY_BASE_URL}/mcp-integrations?page_size={page_size}"
headers = {"x-portkey-api-key": PORTKEY_API_KEY}

try:
    response = requests.get(url, headers=headers, timeout=30)
    print(f"Status Code: {response.status_code}")
    data = response.json()

    integrations = data.get("data", data) if isinstance(data, dict) else data

    if isinstance(integrations, list):
        print(f"\n✅ Found {len(integrations)} integration(s):\n")
        for i, integration in enumerate(integrations, 1):
            print(f"  [{i}] {integration.get('name', 'N/A')}")
            print(f"       ID:        {integration.get('id', 'N/A')}")
            print(f"       URL:       {integration.get('url', 'N/A')}")
            print(f"       Transport: {integration.get('transport', 'N/A')}")
            print(f"       Auth:      {integration.get('auth_type', 'N/A')}")
            print()
    else:
        print(json.dumps(data, indent=2))
except Exception as e:
    print("❌ Error:", str(e))

---
## 9. Fallback Strategy

Portkey's fallback strategy automatically switches to backup LLMs when the primary fails.

### Key concepts
- **Default trigger**: any **non-2xx** status code
- **Custom trigger**: `on_status_codes` array (e.g. `[429]` for rate limits only)
- **Composable**: fallback targets can themselves be load balancers, conditional routers, or nested fallbacks

### Config structure
```json
{
  "strategy": { "mode": "fallback", "on_status_codes": [429, 503] },
  "targets": [
    { "override_params": { "model": "@provider-a/model-a" } },
    { "override_params": { "model": "@provider-b/model-b" } }
  ]
}
```

In [ ]:
# ── Test 9.1: Basic Fallback Between Models ───────────────────────────────────
print("Test 9.1 — Fallback Between Models")
print("-" * 60)

from portkey_ai import Portkey

fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {"override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"}},
        {"override_params": {"model": "@hk-openrouter/mistralai/mistral-7b-instruct"}},
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=fallback_config)

try:
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": "Say hello in one sentence."}],
        max_tokens=50,
    )
    print(f"✅ Model used: {response.model}")
    print(f"   Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
# ── Test 9.2: Fallback on Specific Status Codes (429 rate limit) ──────────────
print("Test 9.2 — Fallback on 429 Only")
print("-" * 60)

fallback_rate_limit_config = {
    "strategy": {"mode": "fallback", "on_status_codes": [429]},
    "targets": [
        {"override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"}},
        {"override_params": {"model": "@hk-openrouter/mistralai/mistral-7b-instruct"}},
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=fallback_rate_limit_config)

try:
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": "What is 2+2?"}],
        max_tokens=50,
    )
    print(f"✅ Model used: {response.model}")
    print(f"   Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
# ── Test 9.3: Multi-tier Fallback (3 models) ──────────────────────────────────
print("Test 9.3 — Multi-tier Fallback (3 models)")
print("-" * 60)

multi_tier_fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {"override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"}},
        {"override_params": {"model": "@hk-openrouter/mistralai/mistral-7b-instruct"}},
        {"override_params": {"model": "@hk-openrouter/google/gemma-2-9b-it"}},
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=multi_tier_fallback_config)

try:
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": "Count to 3."}],
        max_tokens=50,
    )
    print(f"✅ Model used: {response.model}")
    print(f"   Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ Error: {e}")

---
## 10. Automatic Retries

Portkey automatically retries failed requests with exponential backoff.

| Feature | Details |
|---|---|
| Max attempts | **5** |
| Default retry codes | `[429, 500, 502, 503, 504]` |
| Backoff | Exponential: 1s → 2s → 4s → 8s → 16s |
| Custom codes | `on_status_codes` overrides defaults |
| Provider headers | `use_retry_after_headers` respects `Retry-After` |
| Response header | `x-portkey-retry-attempt-count` (>0 = retried, -1 = exhausted, 0 = not configured)

In [ ]:
# ── Test 10.1: Basic Retry (5 attempts) ───────────────────────────────────────
print("Test 10.1 — Basic Retry Config (5 attempts)")
print("-" * 60)

from portkey_ai import Portkey

retry_config = {
    "retry": {"attempts": 5},
    "override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"},
}

client = Portkey(api_key=PORTKEY_API_KEY, config=retry_config)

try:
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": "Say 'retry test passed'."}],
        max_tokens=20,
    )
    print(f"✅ Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
# ── Test 10.2: Retry on Specific Status Codes ─────────────────────────────────
print("Test 10.2 — Retry on [429, 503] Only")
print("-" * 60)

retry_specific_config = {
    "retry": {"attempts": 3, "on_status_codes": [429, 503]},
    "override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"},
}

client = Portkey(api_key=PORTKEY_API_KEY, config=retry_specific_config)

try:
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": "Say 'specific retry test passed'."}],
        max_tokens=20,
    )
    print(f"✅ Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
# ── Test 10.3: Retry with Retry-After Header Support ──────────────────────────
print("Test 10.3 — Retry with use_retry_after_headers")
print("-" * 60)

retry_after_config = {
    "retry": {"attempts": 3, "on_status_codes": [429], "use_retry_after_headers": True},
    "override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"},
}

client = Portkey(api_key=PORTKEY_API_KEY, config=retry_after_config)

try:
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": "Say 'retry-after test passed'."}],
        max_tokens=20,
    )
    print(f"✅ Response: {response.choices[0].message.content}")
    print("   (If 429 occurs, gateway will respect provider's Retry-After header)")
except Exception as e:
    print(f"❌ Error: {e}")

---
## 11. Caching (Simple & Semantic)

Cache LLM responses to serve requests up to **20x faster** and cheaper.

| Mode | How it Works | Supported Routes |
|------|--------------|------------------|
| **Simple** | Exact match on input | All models including image generation |
| **Semantic** | Cosine similarity matching | `/chat/completions`, `/completions` |

### Cache TTL
- Minimum: 60 seconds
- Maximum: 90 days (7,776,000 seconds)
- Default: 7 days (604,800 seconds)

### Per-request options
- `cache_force_refresh`: Bypass cache, fetch fresh response
- `cache_namespace`: Partition cache by custom string (per-user, per-session, etc.)

In [ ]:
# ── Test 11.1: Simple Cache — Measure Speedup ─────────────────────────────────
print("Test 11.1 — Simple Cache (expect faster 2nd request)")
print("-" * 60)

import time
from portkey_ai import Portkey

simple_cache_config = {
    "cache": {"mode": "simple", "max_age": 60},
    "override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"},
}

client = Portkey(api_key=PORTKEY_API_KEY, config=simple_cache_config)
test_msg = [{"role": "user", "content": "What is the capital of France? Answer in one word."}]

start = time.time()
resp1 = client.chat.completions.create(messages=test_msg, max_tokens=10)
t1 = time.time() - start
print(f"  Request 1 (cache miss): {resp1.choices[0].message.content.strip()} [{t1:.3f}s]")

start = time.time()
resp2 = client.chat.completions.create(messages=test_msg, max_tokens=10)
t2 = time.time() - start
print(f"  Request 2 (cache hit):  {resp2.choices[0].message.content.strip()} [{t2:.3f}s]")

if t2 < t1:
    print(f"  ✅ Cache speedup: {t1/t2:.1f}x faster")
else:
    print(f"  ℹ️  No speedup observed (may depend on network conditions)")

In [ ]:
# ── Test 11.2: Cache Force Refresh ─────────────────────────────────────────────
print("Test 11.2 — Cache Force Refresh (bypass existing cache)")
print("-" * 60)

start = time.time()
resp3 = client.with_options(cache_force_refresh=True).chat.completions.create(
    messages=test_msg, max_tokens=10,
)
t3 = time.time() - start
print(f"  Force refresh: {resp3.choices[0].message.content.strip()} [{t3:.3f}s]")
print(f"  (Should be slower than cached response — fetches fresh from LLM)")

In [ ]:
# ── Test 11.3: Cache Namespace (per-user isolation) ────────────────────────────
print("Test 11.3 — Cache Namespace (per-user cache partitioning)")
print("-" * 60)

cache_msg = [{"role": "user", "content": "What is 7 * 8? Answer with just the number."}]

resp_alice = client.with_options(cache_namespace="user-alice").chat.completions.create(
    messages=cache_msg, max_tokens=10,
)
resp_bob = client.with_options(cache_namespace="user-bob").chat.completions.create(
    messages=cache_msg, max_tokens=10,
)
print(f"  User Alice (namespace 'user-alice'): {resp_alice.choices[0].message.content.strip()}")
print(f"  User Bob   (namespace 'user-bob'):   {resp_bob.choices[0].message.content.strip()}")
print("  ✅ Each namespace has its own independent cache partition")

---
## 12. Load Balancing

Distribute traffic across multiple LLMs to prevent bottlenecks.

| Feature | Details |
|---|---|
| Mode | `"loadbalance"` |
| Default weight | `1` (equal distribution) |
| Min weight | `0` (stops traffic without removing from config) |
| Normalization | Weights normalized to 100% automatically |

### Patterns
- **Between providers**: Route to different providers; model comes from request
- **Multiple API keys**: Distribute load across rate limits
- **Cost optimization**: Send most traffic to cheaper models
- **Gradual migration**: Test new models with small percentage

### Sticky sessions
```json
"sticky_session": { "hash_fields": ["metadata.user_id"], "ttl": 3600 }
```
Ensures same user always routes to same target (requires Redis for cross-instance).

In [ ]:
# ── Test 12.1: Weighted Load Balancing (70/30) ─────────────────────────────────
print("Test 12.1 — Weighted Load Balancing (70/30 split)")
print("-" * 60)

from portkey_ai import Portkey

lb_config = {
    "strategy": {"mode": "loadbalance"},
    "targets": [
        {"override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"}, "weight": 0.7},
        {"override_params": {"model": "@hk-openrouter/mistralai/mistral-7b-instruct"}, "weight": 0.3},
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=lb_config)

model_counts = {}
for i in range(5):
    resp = client.chat.completions.create(
        messages=[{"role": "user", "content": f"Say 'hello {i}'."}],
        max_tokens=10,
    )
    model = resp.model
    model_counts[model] = model_counts.get(model, 0) + 1
    print(f"  Request {i+1}: model={model}")

print(f"\n  Distribution: {model_counts}")
print("  (With 70/30 weights, expect ~70% on llama, ~30% on mistral)")

In [ ]:
# ── Test 12.2: Equal Weight Load Balancing ─────────────────────────────────────
print("Test 12.2 — Equal Weight Load Balancing (50/50)")
print("-" * 60)

equal_lb_config = {
    "strategy": {"mode": "loadbalance"},
    "targets": [
        {"override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"}, "weight": 1},
        {"override_params": {"model": "@hk-openrouter/mistralai/mistral-7b-instruct"}, "weight": 1},
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=equal_lb_config)

model_counts = {}
for i in range(6):
    resp = client.chat.completions.create(
        messages=[{"role": "user", "content": f"Say 'test {i}'."}],
        max_tokens=10,
    )
    model = resp.model
    model_counts[model] = model_counts.get(model, 0) + 1

print(f"  Distribution over 6 requests: {model_counts}")
print("  (With equal weights, expect ~50/50 split)")

---
## 13. Conditional Routing

Route requests to different targets based on custom conditions evaluated on the **gateway edge**.

### Condition sources
| Source | Path | Example |
|--------|------|---------|
| Metadata | `metadata.<key>` | `metadata.user_plan` |
| Request params | `params.<key>` | `params.model`, `params.temperature` |
| URL path | `url.pathname` | `/v1/embeddings` |

### Operators
| Comparison | Logical |
|---|---|
| `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte` | `$and`, `$or` (nestable) |
| `$in`, `$nin`, `$regex` | |

### Required fields
- `strategy.conditions` — array of `{ query, then }` objects
- `strategy.default` — target name when no conditions match
- `targets` — array with unique `name` for each target

In [ ]:
# ── Test 13.1: Parameter-based Routing (model alias) ──────────────────────────
print("Test 13.1 — Conditional Routing by Model Alias")
print("-" * 60)

from portkey_ai import Portkey

conditional_config = {
    "strategy": {
        "mode": "conditional",
        "conditions": [
            {
                "query": {"params.model": {"$eq": "fastest"}},
                "then": "fast-target",
            },
            {
                "query": {"params.model": {"$eq": "smartest"}},
                "then": "smart-target",
            },
        ],
        "default": "fast-target",
    },
    "targets": [
        {
            "name": "smart-target",
            "override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"},
        },
        {
            "name": "fast-target",
            "override_params": {"model": "@hk-openrouter/mistralai/mistral-7b-instruct"},
        },
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=conditional_config)

for alias in ["fastest", "smartest", "unknown-alias"]:
    try:
        resp = client.chat.completions.create(
            model=alias,
            messages=[{"role": "user", "content": "Say hello."}],
            max_tokens=10,
        )
        print(f"  model='{alias}' → routed to: {resp.model}")
    except Exception as e:
        print(f"  model='{alias}' → Error: {str(e)[:120]}")

In [ ]:
# ── Test 13.2: Metadata-based Routing ─────────────────────────────────────────
print("Test 13.2 — Conditional Routing by Metadata (user_plan)")
print("-" * 60)

metadata_routing_config = {
    "strategy": {
        "mode": "conditional",
        "conditions": [
            {
                "query": {"metadata.user_plan": {"$eq": "paid"}},
                "then": "premium-target",
            },
            {
                "query": {"metadata.user_plan": {"$eq": "free"}},
                "then": "basic-target",
            },
        ],
        "default": "basic-target",
    },
    "targets": [
        {
            "name": "premium-target",
            "override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"},
        },
        {
            "name": "basic-target",
            "override_params": {"model": "@hk-openrouter/mistralai/mistral-7b-instruct"},
        },
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=metadata_routing_config)

for plan in ["paid", "free"]:
    try:
        resp = client.with_options(
            metadata={"user_plan": plan}
        ).chat.completions.create(
            messages=[{"role": "user", "content": "Say hello."}],
            max_tokens=10,
        )
        print(f"  user_plan='{plan}' → routed to: {resp.model}")
    except Exception as e:
        print(f"  user_plan='{plan}' → Error: {str(e)[:120]}")

---
## 14. Canary Testing

Uses the same technique as **load balancing** but with **heavily skewed weights** to safely test new models in production on a small percentage of traffic.

### Pattern
Send 95% to the proven model and 5% to the candidate model. Monitor via Portkey analytics dashboards for cost, latency, errors, and feedback impact.

In [ ]:
# ── Test 14.1: Canary Test (95/5 split) ────────────────────────────────────────
print("Test 14.1 — Canary Testing (95% primary, 5% canary)")
print("-" * 60)

from portkey_ai import Portkey

canary_config = {
    "strategy": {"mode": "loadbalance"},
    "targets": [
        {"override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"}, "weight": 0.95},
        {"override_params": {"model": "@hk-openrouter/mistralai/mistral-7b-instruct"}, "weight": 0.05},
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=canary_config)

model_counts = {}
for i in range(10):
    resp = client.chat.completions.create(
        messages=[{"role": "user", "content": f"Canary test request {i}"}],
        max_tokens=10,
    )
    model = resp.model
    model_counts[model] = model_counts.get(model, 0) + 1

print(f"  Distribution over 10 requests: {model_counts}")
print("  (Expect ~95% on primary llama, ~5% on canary mistral)")

---
## 15. Combined / Nested Strategies

Portkey strategies are **fully composable** — any strategy can nest inside any other:

| Pattern | Use Case |
|---|---|
| Fallback → Load Balancer | Cluster failover — individual failures stay in-cluster |
| Load Balancer → Fallback | Per-slot resilience — each slot has its own backup |
| Conditional → Load Balancer | Model alias → multi-provider distribution |
| Fallback + Retry + Cache | Kitchen sink — full resilience + cost savings |

In [ ]:
# ── Test 15.1: Load Balance with Nested Fallback ──────────────────────────────
print("Test 15.1 — Load Balance + Nested Fallback")
print("-" * 60)

from portkey_ai import Portkey

nested_config = {
    "strategy": {"mode": "loadbalance"},
    "targets": [
        {
            "override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"},
            "weight": 0.5,
        },
        {
            "strategy": {"mode": "fallback", "on_status_codes": [429]},
            "targets": [
                {"override_params": {"model": "@hk-openrouter/mistralai/mistral-7b-instruct"}},
                {"override_params": {"model": "@hk-openrouter/google/gemma-2-9b-it"}},
            ],
            "weight": 0.5,
        },
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=nested_config)

model_counts = {}
for i in range(6):
    resp = client.chat.completions.create(
        messages=[{"role": "user", "content": f"Nested test {i}"}],
        max_tokens=10,
    )
    model = resp.model
    model_counts[model] = model_counts.get(model, 0) + 1
    print(f"  Request {i+1}: {model}")

print(f"\n  Distribution: {model_counts}")
print("  (50% direct llama, 50% to fallback group [mistral → gemma])")

In [ ]:
# ── Test 15.2: Kitchen Sink — Fallback + Retry + Cache ─────────────────────────
print("Test 15.2 — Fallback + Retry + Cache Combined")
print("-" * 60)

kitchen_sink_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {"override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"}},
        {"override_params": {"model": "@hk-openrouter/mistralai/mistral-7b-instruct"}},
    ],
    "retry": {"attempts": 3, "on_status_codes": [429, 500]},
    "cache": {"mode": "simple", "max_age": 60},
}

client = Portkey(api_key=PORTKEY_API_KEY, config=kitchen_sink_config)

try:
    resp = client.chat.completions.create(
        messages=[{"role": "user", "content": "What is 1+1? Answer with just the number."}],
        max_tokens=10,
    )
    print(f"  ✅ Model: {resp.model}")
    print(f"     Response: {resp.choices[0].message.content}")
    print("     Config active: fallback (2 models) + retry (3 attempts on 429/500) + simple cache (60s)")
except Exception as e:
    print(f"  ❌ Error: {e}")

---
## 16. Guardrails with Configs (Advanced Patterns)

Combine guardrails with other gateway features for production-grade orchestration.

### Guardrail status codes (sync guardrails, `async=false`)
| Verdict | Deny | Status Code | Behavior |
|---------|------|-------------|----------|
| PASS | any | **200** | Request processed normally |
| FAIL | false | **246** | Request still processed (soft fail, logged) |
| FAIL | true | **446** | Request blocked (hard fail) |

### Orchestration patterns
- **Retry on guardrail fail**: `retry.on_status_codes: [246]` — re-run LLM on output guardrail failure
- **Fallback on guardrail fail**: `strategy.on_status_codes: [246, 446]` — try next provider

> Replace `pg-lowerc-160e3e` below with your actual guardrail ID from the Portkey dashboard.

In [ ]:
# ── Test 16.1: Guardrails + Retry on Output Fail ──────────────────────────────
print("Test 16.1 — Output Guardrail + Retry on Fail (246)")
print("-" * 60)

from portkey_ai import Portkey

GUARDRAIL_ID = "pg-lowerc-160e3e"  # Replace with your guardrail ID

guardrail_retry_config = {
    "retry": {"on_status_codes": [246], "attempts": 3},
    "override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"},
    "output_guardrails": [GUARDRAIL_ID],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=guardrail_retry_config)

try:
    resp = client.chat.completions.create(
        messages=[{"role": "user", "content": "Tell me a short joke."}],
        max_tokens=100,
    )
    print(f"  ✅ Response (after guardrail check): {resp.choices[0].message.content[:200]}")
except Exception as e:
    print(f"  ❌ Blocked after retries: {str(e)[:200]}")

In [ ]:
# ── Test 16.2: Guardrails + Fallback on Input Fail ────────────────────────────
print("Test 16.2 — Input Guardrail + Fallback on Fail (246/446)")
print("-" * 60)

guardrail_fallback_config = {
    "strategy": {"mode": "fallback", "on_status_codes": [246, 446]},
    "targets": [
        {"override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"}},
        {"override_params": {"model": "@hk-openrouter/mistralai/mistral-7b-instruct"}},
    ],
    "input_guardrails": [GUARDRAIL_ID],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=guardrail_fallback_config)

try:
    resp = client.chat.completions.create(
        messages=[{"role": "user", "content": "Hello, can you help me?"}],
        max_tokens=50,
    )
    print(f"  ✅ Model: {resp.model}")
    print(f"     Response: {resp.choices[0].message.content[:200]}")
except Exception as e:
    print(f"  ❌ Error: {str(e)[:200]}")

---
## 17. Metadata & Tracing

### Metadata
Attach custom contextual information to requests for filtering, analytics, and cost attribution.

| Key | Purpose |
|-----|---------|
| `_user` | Powers user-level analytics (auto-copied from `user` in request body) |
| `environment` | Differentiate dev/staging/prod |
| `feature` | Track which product feature uses AI |
| `session_id` | Group related requests |

- All values must be **strings**, max **128 characters**
- Set via `with_options(metadata={...})` or in constructor

### Tracing
- `trace_id`: Correlate all requests in a workflow/session
- `span_id`, `parent_span_id`: Fine-grained spans within a trace
- W3C `traceparent` / `baggage` headers also supported

In [ ]:
# ── Test 17.1: Metadata Tagging ────────────────────────────────────────────────
print("Test 17.1 — Metadata Tagging")
print("-" * 60)

from portkey_ai import Portkey

client = Portkey(
    api_key=PORTKEY_API_KEY,
    config={"override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"}},
)

resp = client.with_options(
    metadata={
        "_user": "test-user-42",
        "environment": "testing",
        "feature": "comprehensive-notebook",
        "session_id": "sess-001",
    }
).chat.completions.create(
    messages=[{"role": "user", "content": "Say 'metadata test passed'."}],
    max_tokens=20,
)
print(f"  ✅ Response: {resp.choices[0].message.content}")
print("  → Check Portkey dashboard logs, filter by _user='test-user-42'")

In [ ]:
# ── Test 17.2: Trace ID Correlation ────────────────────────────────────────────
print("Test 17.2 — Trace ID Correlation (multi-step workflow)")
print("-" * 60)

from uuid import uuid4

trace = f"notebook-trace-{uuid4().hex[:8]}"

client = Portkey(
    api_key=PORTKEY_API_KEY,
    trace_id=trace,
    config={"override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"}},
)

resp1 = client.chat.completions.create(
    messages=[{"role": "user", "content": "Step 1: What is Python?"}],
    max_tokens=30,
)
resp2 = client.chat.completions.create(
    messages=[{"role": "user", "content": "Step 2: What is JavaScript?"}],
    max_tokens=30,
)
print(f"  Trace ID: {trace}")
print(f"  Step 1: {resp1.choices[0].message.content[:80]}...")
print(f"  Step 2: {resp2.choices[0].message.content[:80]}...")
print("  ✅ Both requests share the same trace_id in Portkey logs")

---
## 18. Streaming Chat Completions

Test streaming responses via the Portkey SDK. Streaming works with all gateway features (fallbacks, load balancing, caching, guardrails).

> **Note**: For streaming + guardrails, only **input guardrails** are supported on the output path. Set `x-portkey-strict-open-ai-compliance: false` to access `hook_results` in stream chunks.

In [ ]:
# ── Test 18.1: Basic Streaming ─────────────────────────────────────────────────
print("Test 18.1 — Streaming Chat Completion")
print("-" * 60)

from portkey_ai import Portkey

client = Portkey(
    api_key=PORTKEY_API_KEY,
    config={"override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"}},
)

print("  Response: ", end="", flush=True)

stream = client.chat.completions.create(
    messages=[{"role": "user", "content": "Count from 1 to 5, separated by commas."}],
    max_tokens=50,
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta
    if delta.content:
        print(delta.content, end="", flush=True)

print("\n  ✅ Stream complete.")

In [ ]:
# ── Test 18.2: Streaming + Fallback ────────────────────────────────────────────
print("Test 18.2 — Streaming + Fallback")
print("-" * 60)

stream_fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {"override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"}},
        {"override_params": {"model": "@hk-openrouter/mistralai/mistral-7b-instruct"}},
    ],
}

client = Portkey(api_key=PORTKEY_API_KEY, config=stream_fallback_config)

print("  Response: ", end="", flush=True)

stream = client.chat.completions.create(
    messages=[{"role": "user", "content": "Say 'streaming fallback works' in one sentence."}],
    max_tokens=30,
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta
    if delta.content:
        print(delta.content, end="", flush=True)

print("\n  ✅ Stream with fallback complete.")

---
## 19. Virtual Keys / Model Catalog

Virtual keys abstract provider credentials behind a slug, enabling provider-agnostic code.

### Format
`@provider-slug/model-name` — automatically routes to the correct provider.

### SDK Management API
| Method | Description |
|--------|-------------|
| `virtual_keys.create()` | Create a new virtual key (name, provider, key, rate_limits) |
| `virtual_keys.list()` | List all virtual keys in workspace |
| `virtual_keys.retrieve(slug)` | Get details of a specific virtual key |
| `virtual_keys.update(slug)` | Update virtual key settings |
| `virtual_keys.delete(slug)` | Delete a virtual key |

In [ ]:
# ── Test 19.1: List Virtual Keys ───────────────────────────────────────────────
print("Test 19.1 — List Virtual Keys in Workspace")
print("-" * 60)

from portkey_ai import Portkey

client = Portkey(api_key=PORTKEY_API_KEY)

try:
    vkeys = client.virtual_keys.list()
    if hasattr(vkeys, 'data') and vkeys.data:
        for vk in vkeys.data[:5]:
            slug = getattr(vk, 'slug', 'N/A')
            name = getattr(vk, 'name', 'N/A')
            provider = getattr(vk, 'provider', 'N/A')
            print(f"  [{slug}] {name} — provider: {provider}")
        if len(vkeys.data) > 5:
            print(f"  ... and {len(vkeys.data) - 5} more")
        print(f"\n  ✅ Total virtual keys: {len(vkeys.data)}")
    else:
        print("  ℹ️  No virtual keys found or different response structure.")
        print(f"     Raw response: {str(vkeys)[:300]}")
except Exception as e:
    print(f"  ❌ Error: {str(e)[:200]}")

---
## 20. Feedback API

Attach human or automated feedback to requests using `trace_id` for evaluation and quality tracking.

### SDK methods
| Method | Description |
|--------|-------------|
| `feedback.create()` | Submit feedback for a single trace |
| `feedback.bulk_create()` | Submit feedback for multiple traces |
| `feedback.update()` | Update existing feedback |

### Parameters
- `trace_id` — links feedback to a specific request
- `value` — numerical score (e.g. 1-5)
- `weight` — importance weight
- `metadata` — additional context (source, quality label, etc.)

In [ ]:
# ── Test 20.1: Submit Feedback for a Request ──────────────────────────────────
print("Test 20.1 — Submit Feedback via Trace ID")
print("-" * 60)

from portkey_ai import Portkey
from uuid import uuid4

client = Portkey(api_key=PORTKEY_API_KEY)
trace = f"feedback-test-{uuid4().hex[:8]}"

# First, make a request with the trace ID
resp = Portkey(
    api_key=PORTKEY_API_KEY,
    trace_id=trace,
    config={"override_params": {"model": "@hk-openrouter/meta-llama/llama-3.3-70b-instruct"}},
).chat.completions.create(
    messages=[{"role": "user", "content": "Tell me a fun fact about space."}],
    max_tokens=60,
)

print(f"  Trace ID: {trace}")
print(f"  LLM Response: {resp.choices[0].message.content[:120]}")

# Then, submit feedback for that request
try:
    fb = client.feedback.create(
        trace_id=trace,
        value=5,
        weight=1,
        metadata={"source": "notebook-test", "quality": "good"},
    )
    print(f"  ✅ Feedback submitted (value=5, weight=1)")
    print(f"     Check Portkey logs → Feedback & Guardrails tab for trace: {trace}")
except Exception as e:
    print(f"  ❌ Feedback error: {str(e)[:200]}")

---
## 21. Prompt Templates (Prompt API)

Portkey's Prompt Engineering Studio lets you version, test, and deploy prompt templates.

### SDK methods
| Method | Description |
|--------|-------------|
| `prompts.completions.create()` | Render template + run LLM completion |
| `prompts.render()` | Render template without calling LLM |
| `prompts.list()` | List all prompts in workspace |
| `prompts.retrieve(id)` | Get prompt details |

### Usage
```python
completion = portkey.prompts.completions.create(
    prompt_id="YOUR_PROMPT_ID",
    variables={"user_input": "Hello world"},
    max_tokens=250
)
```

In [ ]:
# ── Test 21.1: List Prompts ────────────────────────────────────────────────────
print("Test 21.1 — List Available Prompt Templates")
print("-" * 60)

from portkey_ai import Portkey

client = Portkey(api_key=PORTKEY_API_KEY)

try:
    prompts_list = client.prompts.list()
    if hasattr(prompts_list, 'data') and prompts_list.data:
        for p in prompts_list.data[:5]:
            pid = getattr(p, 'id', 'N/A')
            name = getattr(p, 'name', 'N/A')
            print(f"  [{pid}] {name}")
        if len(prompts_list.data) > 5:
            print(f"  ... and {len(prompts_list.data) - 5} more")
        print(f"\n  ✅ Total prompts: {len(prompts_list.data)}")

        # If prompts exist, try running the first one
        first_prompt = prompts_list.data[0]
        prompt_id = getattr(first_prompt, 'id', None)
        if prompt_id:
            print(f"\n  Attempting completion with prompt '{prompt_id}'...")
            try:
                completion = client.prompts.completions.create(
                    prompt_id=prompt_id,
                    max_tokens=50,
                )
                print(f"  ✅ Prompt completion: {completion.choices[0].message.content[:150]}")
            except Exception as e:
                print(f"  ⚠️  Completion error (may need variables): {str(e)[:150]}")
    else:
        print("  ℹ️  No prompts found. Create one in the Portkey dashboard first.")
        print(f"     Raw response: {str(prompts_list)[:300]}")
except Exception as e:
    print(f"  ❌ Error: {str(e)[:200]}")

---
## Summary

| # | Section | Feature | Status | Notes |
|---|---------|---------|--------|-------|
| 1 | Setup | Dependencies & env | ✅ | `portkey-ai`, `fastmcp`, `python-dotenv` |
| 2 | Basic Chat Completion | `@virtual-key/model` | ✅ | Uses `@hk-openrouter` virtual key |
| 3 | Inline Guardrail Config | Dashboard guardrail via config | ✅ | References `pg-lowerc-160e3e` |
| 4 | Guardrail Tests | `default.regexMatch` — `rule` vs `regex` | ✅ | 3 tests: block, allow, wrong key |
| 5 | Self-Hosted Gateway | Health, chat, streaming via REST | ⚠️ | Requires local gateway on port 8787 |
| 6 | MCP Server | FastMCP tools, resources, prompts | ✅ | Run `python test_mcp_server.py` to start |
| 7 | MCP Registry | Register MCP with Portkey | ✅ | Public URL needed for cloud |
| 8 | MCP Listing | List MCP integrations | ✅ | Lists all workspace integrations |
| **9** | **Fallback Strategy** | Between models, rate-limit, multi-tier | ✅ | 3 tests: basic, 429-only, 3-model chain |
| **10** | **Automatic Retries** | Basic, specific codes, retry-after | ✅ | 3 tests: 5-attempt, custom codes, headers |
| **11** | **Caching** | Simple cache, force refresh, namespace | ✅ | 3 tests: speedup, bypass, per-user |
| **12** | **Load Balancing** | Weighted distribution | ✅ | 2 tests: 70/30 split, equal weight |
| **13** | **Conditional Routing** | Param-based, metadata-based | ✅ | 2 tests: model alias, user_plan routing |
| **14** | **Canary Testing** | 95/5 traffic split | ✅ | Load balance with skewed weights |
| **15** | **Combined Strategies** | Nested LB+fallback, kitchen sink | ✅ | 2 tests: nested config, FB+retry+cache |
| **16** | **Guardrails + Configs** | Retry on fail, fallback on fail | ✅ | 2 tests: 246 retry, 246/446 fallback |
| **17** | **Metadata & Tracing** | Tags, trace_id, user tracking | ✅ | 2 tests: metadata, trace correlation |
| **18** | **Streaming** | Basic stream, stream + fallback | ✅ | 2 tests: SSE chunks, stream w/ fallback |
| **19** | **Virtual Keys** | List/manage virtual keys | ✅ | SDK management API |
| **20** | **Feedback API** | Submit feedback via trace_id | ✅ | Create + attach feedback to requests |
| **21** | **Prompt Templates** | List prompts, render, completions | ✅ | Prompt Engineering Studio API |

---
### Useful Links
- [Portkey Documentation](https://docs.portkey.ai)
- [Portkey AI Gateway (GitHub)](https://github.com/Portkey-AI/gateway)
- [FastMCP Documentation](https://gofastmcp.com)
- [Portkey Dashboard](https://app.portkey.ai)